# Kaggle ↔ Colab 任务桥(GitHub 托管版 · 支持换账号登录)

**换账号用法(你的情况):** Colab 用**新账号**登录也能跑——本笔记本放在公开 GitHub 仓库里,
任何 Google 账号都可打开;任务信箱仍在**旧账号**的私有 Drive 文件夹里,挂载 Drive 时选它即可。

**运行前(共 2 个一次性动作):**
1. **挂载 Drive 的授权弹窗里点「之前那个账号」**(信箱文件夹 `Colab Notebooks/Kaggle-Colab-Bridge` 在它的 Drive 里)。
2. 左侧 🔑 **Secrets** 新建 `KAGGLE_API_TOKEN`,粘贴 Kaggle 密钥(`KGAT_…` 或 `sk-…` 裸 token 都认;
   旧版 `KAGGLE_USERNAME`+`KAGGLE_KEY` 两件套也行),打开 **Notebook access**。密钥只存 Secrets,
   **不要**发到聊天、单元格或 Drive。
3. 需要 GPU 时再改运行时类型;首次验证 CPU 也行(冒烟任务会自动推 Kaggle GPU kernel)。

**生命周期:** worker 前台运行最长 2 小时;执行 `inbox/` 里助手提交的受信任 Python 任务,
日志与小文件回传 `results/`。停止:Runtime → Interrupt,或在 Drive 的 `control/` 新建 `stop` 文件。
任务最大 256 KiB;回传最多 20 个文件、每个 20 MiB、共 50 MiB。任务代码不是沙箱隔离的,
请保持信箱文件夹私有。

参考:[Google Colab MCP](https://github.com/googlecolab/colab-mcp) · [Kaggle 官方 CLI](https://github.com/Kaggle/kaggle-cli) ·
[Kaggle/Colab 数据交换先例](https://github.com/IMOKURI/kaggle_on_google_colab)。本笔记本使用 Drive 中转。

## 1 · 挂载并核对私有信箱(弹窗里点**之前那个账号**)

只读取这个固定文件夹,不改动其他内容。

In [ ]:
from pathlib import Path
import json
from google.colab import drive

drive.mount("/content/drive")
BRIDGE_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/Kaggle-Colab-Bridge")
marker = json.loads((BRIDGE_ROOT / "bridge.json").read_text(encoding="utf-8"))
assert marker.get("bridge_id") == "kaggle-colab-drive-v1", "Wrong bridge marker"
assert marker.get("drive_folder_id") == "1zauC0S_rrR2SnW-gX5phMAFVAs4bB38g", "Wrong Drive folder"
for name in ("inbox", "claims", "results", "status", "control"):
    (BRIDGE_ROOT / name).mkdir(exist_ok=True)
assert not (BRIDGE_ROOT / "control" / "stop").exists(), "Remove control/stop before starting"
assert not (BRIDGE_ROOT / "control" / "worker.lock").exists(), "Stop the old worker; inspect any stale lock first"
print("Private bridge directory verified:", BRIDGE_ROOT)

## 2 · 安装官方 Kaggle CLI

固定到核查过的版本;此格不验证登录、不显示密钥。

In [ ]:
%pip install -q "kaggle==2.2.5"

## 3 · 读取 Secrets(当前登录账号的 🔑 面板)

`KAGGLE_API_TOKEN`(KGAT_… 或裸 token)或 `KAGGLE_USERNAME`+`KAGGLE_KEY`;务必打开 Notebook access。

In [ ]:
# Secrets stay in notebook memory and are passed only to trusted job subprocesses.
# Never write their values into Drive, Git, notebook outputs, or artifacts.
from google.colab import userdata
import re

_BRIDGE_SECRETS = {}
def optional_secret(name):
    try:
        value = userdata.get(name)
        return value.strip() if value else ""
    except Exception as exc:
        print(f"{name}: unavailable ({type(exc).__name__}). Check Secrets / Notebook access if intended.")
        return ""

token = optional_secret("KAGGLE_API_TOKEN")
username = optional_secret("KAGGLE_USERNAME")
key = optional_secret("KAGGLE_KEY")
if token:
    if not re.fullmatch(r"(KGAT_[0-9a-fA-F]{32}|[A-Za-z0-9_\-]{20,})", token):
        raise ValueError("KAGGLE_API_TOKEN must be a raw Kaggle token (KGAT_… or a bare API token). "
                         "For legacy credentials use KAGGLE_USERNAME + KAGGLE_KEY instead.")
    _BRIDGE_SECRETS["KAGGLE_API_TOKEN"] = token
elif username and key:
    if key.startswith(("KGAT_", "KGRT_")):
        raise ValueError("A KGAT token belongs in KAGGLE_API_TOKEN, not KAGGLE_KEY.")
    _BRIDGE_SECRETS.update(KAGGLE_USERNAME=username, KAGGLE_KEY=key)
elif username or key:
    raise ValueError("Legacy authentication needs BOTH KAGGLE_USERNAME and KAGGLE_KEY.")
del token, username, key
print("Kaggle credential source:", "token secret" if "KAGGLE_API_TOKEN" in _BRIDGE_SECRETS
      else "legacy secret pair" if _BRIDGE_SECRETS else "not configured; Colab-only test")

## 4 · 启动接收任务(worker)

保持此格运行。助手会读取 `status/worker.json` 和 `results/`。首次连通探针已在 `inbox/`,
`kaggle-t4-smoke` 任务(经 kaggle CLI 推 2×T4 kernel)也在排队。不要同时运行两个 worker。

In [ ]:
"""User-started, private Drive mailbox worker for Colab (bridge v2).

Exploratory bridge, not a security sandbox. Jobs are trusted Python code authorized
by the person who starts this cell. Credentials are passed to job subprocesses only
in memory and redacted from logs. Stop with Interrupt execution or control/stop.
"""
from __future__ import annotations
import json
import os
from pathlib import Path
import re
import subprocess
import sys
import time
import uuid
from datetime import datetime, timezone

JOB_ID = re.compile(r"^[A-Za-z0-9][A-Za-z0-9_-]{0,79}$")
MAX_JOB_BYTES = 256 * 1024
MAX_LOG_BYTES = 32 * 1024
MAX_ARTIFACT_FILES = 20
MAX_ARTIFACT_BYTES = 20 * 1024 * 1024
MAX_ARTIFACT_TOTAL_BYTES = 50 * 1024 * 1024


def now() -> str:
    return datetime.now(timezone.utc).isoformat()


def clean(value, secrets: dict[str, str]) -> str:
    """Redact explicitly supplied credentials; never print the credential map."""
    text = str(value)
    for secret in sorted(set(secrets.values()), key=len, reverse=True):
        if secret:
            text = text.replace(secret, "[REDACTED]")
    return text


def atomic_json(path: Path, value: dict) -> None:
    temporary = path.with_name(f".{path.name}.{uuid.uuid4().hex}.tmp")
    with temporary.open("x", encoding="utf-8") as stream:
        json.dump(value, stream, ensure_ascii=False, indent=2)
    os.replace(temporary, path)


def bounded_log(text: str) -> dict:
    if len(text) <= MAX_LOG_BYTES:
        return {"text": text, "truncated": False}
    half = MAX_LOG_BYTES // 2
    return {"text": text[:half] + "\n[... middle omitted ...]\n" + text[-half:], "truncated": True}


def run_worker(root: Path, secrets: dict[str, str], max_seconds: float = 7200, poll_seconds: float = 5) -> None:
    inbox, claims, results, status, control = (root / n for n in ("inbox", "claims", "results", "status", "control"))
    for folder in (inbox, claims, results, status, control):
        folder.mkdir(exist_ok=True)
    stop = control / "stop"
    lock = control / "worker.lock"
    session = uuid.uuid4().hex
    atomic_json(lock, {"session": session, "pid": os.getpid(), "started_at": now()})
    state = {"session": session, "state": "running", "started_at": now(),
             "heartbeat_at": now(), "current_job": None, "completed_jobs": 0}

    def heartbeat() -> None:
        state["heartbeat_at"] = now()
        atomic_json(status / "worker.json", dict(state))

    heartbeat()
    print(f"worker {session}: polling {inbox} every {poll_seconds}s, session cap {max_seconds}s", flush=True)
    deadline = time.monotonic() + max_seconds
    last_heartbeat = time.monotonic()
    try:
        while time.monotonic() < deadline and not stop.exists():
            for job_file in sorted(inbox.glob("*.json")):
                if time.monotonic() >= deadline or stop.exists():
                    break
                if job_file.name.startswith("."):
                    continue
                try:
                    job = json.loads(job_file.read_text(encoding="utf-8"))
                except (OSError, ValueError) as exc:
                    print(f"skip unreadable {job_file.name}: {type(exc).__name__}", flush=True)
                    continue
                jid = str(job.get("id", ""))
                if not JOB_ID.fullmatch(jid):
                    print(f"skip {job_file.name}: invalid job id", flush=True)
                    continue
                code = job.get("code", "")
                if not isinstance(code, str) or not code.strip():
                    print(f"skip {jid}: empty code", flush=True)
                    continue
                if len(code.encode("utf-8")) > MAX_JOB_BYTES:
                    print(f"skip {jid}: code exceeds {MAX_JOB_BYTES} bytes", flush=True)
                    continue
                claim = claims / f"{jid}.json"
                try:
                    with claim.open("x", encoding="utf-8") as stream:
                        json.dump({"session": session, "job": jid, "claimed_at": now()}, stream)
                except FileExistsError:
                    continue  # already claimed (possibly by an earlier session)
                print(f"[{jid}] claimed", flush=True)
                state["current_job"] = jid
                heartbeat()
                job_dir = results / jid
                job_dir.mkdir(exist_ok=True)
                artifacts = job_dir / "artifacts"
                artifacts.mkdir(exist_ok=True)
                entry = Path(f"/tmp/bridge_job_{jid}.py")
                entry.write_text(code, encoding="utf-8")
                job_env = os.environ.copy()
                job_env["BRIDGE_ARTIFACT_DIR"] = str(artifacts)
                job_env.update(secrets)
                timeout = int(job.get("timeout_seconds", 600) or 600)
                started = time.time()
                exit_code, error, raw_log = None, None, ""
                try:
                    proc = subprocess.run([sys.executable, str(entry)], env=job_env,
                                          capture_output=True, text=True, timeout=timeout)
                    exit_code = proc.returncode
                    raw_log = (proc.stdout or "") + ("\n--- STDERR ---\n" + (proc.stderr or "") if proc.stderr else "")
                except subprocess.TimeoutExpired as exc:
                    error = f"timeout after {timeout}s"
                    out = exc.stdout or ""
                    err = exc.stderr or ""
                    if isinstance(out, bytes):
                        out = out.decode("utf-8", "replace")
                    if isinstance(err, bytes):
                        err = err.decode("utf-8", "replace")
                    raw_log = str(out) + "\n--- STDERR ---\n" + str(err)
                finally:
                    entry.unlink(missing_ok=True)
                log = bounded_log(clean(raw_log, secrets))
                kept, total = [], 0
                for artifact in sorted(artifacts.rglob("*")):
                    if not artifact.is_file():
                        continue
                    size = artifact.stat().st_size
                    if (len(kept) >= MAX_ARTIFACT_FILES or size > MAX_ARTIFACT_BYTES
                            or total + size > MAX_ARTIFACT_TOTAL_BYTES):
                        continue
                    kept.append({"name": str(artifact.relative_to(artifacts)), "bytes": size})
                    total += size
                atomic_json(job_dir / "result.json", {
                    "id": jid, "session": session, "claimed_at": now(), "finished_at": now(),
                    "wall_sec": round(time.time() - started, 2), "exit_code": exit_code,
                    "error": clean(error, secrets) if error else None,
                    "log": log, "artifacts": kept,
                })
                print(f"[{jid}] finished exit={exit_code} artifacts={len(kept)}", flush=True)
                state["completed_jobs"] += 1
                state["current_job"] = None
                heartbeat()
            if time.monotonic() - last_heartbeat >= 30:
                heartbeat()
                last_heartbeat = time.monotonic()
            time.sleep(min(poll_seconds, max(0.0, deadline - time.monotonic())))
        state["state"] = "stopped" if stop.exists() else "expired"
    except KeyboardInterrupt:
        state["state"] = "interrupted"
        print("interrupted by user", flush=True)
    except Exception as exc:
        state["state"] = "error"
        state["error"] = clean(exc, secrets)
        raise
    finally:
        state["current_job"] = None
        heartbeat()
        try:
            owner = json.loads(lock.read_text(encoding="utf-8")).get("session")
            if owner == session:
                lock.unlink(missing_ok=True)
        except (OSError, ValueError):
            print("Could not remove this worker's lock; inspect it before restarting.", flush=True)
        print("Worker stopped. Remote Kaggle jobs already submitted are not cancelled.", flush=True)


# Explicit, foreground, finite session. No automatic restart or hidden daemon.
run_worker(BRIDGE_ROOT, _BRIDGE_SECRETS, max_seconds=7200, poll_seconds=5)

## 5 · 停止以后再说

正常情况第 4 格会一直运行。中断后再运行本格防误重启。

In [ ]:
(BRIDGE_ROOT / "control" / "stop").write_text("stopped by notebook user", encoding="utf-8")
print("Stop requested. If a worker is running, it will stop after terminating the current job.")